# Chapter 4: Optimization


<!-- Macro definitions for MathJax, mirroring book.tex -->
$$
\newcommand{\bm}[1]{\boldsymbol{#1}}
\newcommand{\Det}[1]{|\boldsymbol{#1}|}
\newcommand{\bigO}{\mathcal{O}}
\newcommand{\var}{\mathrm{Var}}
\newcommand{\cov}{\mathrm{Cov}}
\newcommand{\Prob}{\mathrm{Prob}}
\newcommand{\mean}[1]{\langle #1 \rangle}
$$

Almost every problem in machine learning begins with a data set $\bm{X}$, a
model $g(\bm{\theta})$ depending on parameters $\bm{\theta}$, and a cost
function $C(\bm{X},g(\bm{\theta}))$ measuring how badly the model explains the
data.  The model is fitted by finding the $\bm{\theta}$ that minimises the
cost.  In Chapter 3 we were fortunate: the cost was quadratic,
its gradient vanished at a point we could write down, and the minimisation
reduced to a linear system.  That good fortune does not survive contact with
logistic regression, and it certainly does not survive neural networks.  For
everything that follows in this book we must find minima numerically.

This chapter builds the tools.  We begin with the question of when a minimum
is worth looking for at all -- convexity -- and with Newton's method, which is
the ideal we shall spend the rest of the chapter approximating cheaply.  We
then develop gradient descent, using linear regression as a test bed precisely
because we already know the answer, and we identify exactly what makes plain
gradient descent slow: the conditioning of the Hessian, a quantity we have
already met twice, in Section *Vector and matrix norms* as a numerical diagnosis and in
Section *Statistical properties of the least-squares estimator* as a statistical one.  Everything after that --
momentum, stochastic gradients, AdaGrad, RMSProp and Adam -- is an attempt to
defeat that one number without paying the cost of computing the Hessian.

The reader who wants a single organising idea should hold on to this: gradient
descent takes the same step in every direction, but the cost function does not
curve equally in every direction, and the whole art lies in fixing that
mismatch.


## Convexity

Ideally we want our cost function to be convex, and it is worth stating
precisely what that buys us.

We first need convex sets.  A set $C\subset\mathbb{R}^{n}$ is *convex* if,
for all $\bm{x}$ and $\bm{y}$ in $C$ and all $t\in(0,1)$, the point
$(1-t)\bm{x}+t\bm{y}$ also belongs to $C$; geometrically, every point on the
line segment joining two points of $C$ lies in $C$.  The convex subsets of
$\mathbb{R}$ are the intervals; examples in $\mathbb{R}^{2}$ include the
regular polygons and the discs.

**Convex functions.** 
Let $X\subset\mathbb{R}^{n}$ be a convex set and $f:X\rightarrow\mathbb{R}$
continuous.  Then $f$ is *convex* if

$$
f\left(t\bm{x}_1+(1-t)\bm{x}_2\right)
   \le t f(\bm{x}_1) + (1-t) f(\bm{x}_2)\tag{4.1}
$$

for all $\bm{x}_1,\bm{x}_2\in X$ and all $t\in[0,1]$.  Replacing $\le$ by a
strict inequality, with $\bm{x}_1\neq\bm{x}_2$ and $t\in(0,1)$, defines a
*strictly convex* function.  For a function of one variable the condition
says that the chord joining $f(x_1)$ and $f(x_2)$ lies above the graph on the
whole interval $[x_1,x_2]$.

**First-order condition.** 
Suppose $f$ is differentiable.  Then $f$ is convex if and only if its domain
$D_f$ is convex and

$$
f(\bm{y}) \ge f(\bm{x}) + \nabla f(\bm{x})^{T}(\bm{y}-\bm{x})\tag{4.2}
$$

for all $\bm{x},\bm{y}\in D_f$.  In words: the first-order Taylor expansion at
any point is a global underestimator of the function.  Drawing the tangent to
$f(x)=x^{2}+1$ at any point and observing that it lies everywhere below the
parabola is enough to make the statement believable.

**Second-order condition.** 
If $f$ is twice differentiable, so that the Hessian exists everywhere, then
$f$ is convex if and only if $D_f$ is convex and the Hessian is positive
semi-definite at every point of $D_f$.  For one variable this reduces to
$f''(x)\ge0$: non-negative curvature everywhere.  This condition is the useful
one in practice, because it gives a procedure rather than a definition.  Proofs
of both conditions may be found in Boyd and Vandenberghe [boyd2004].

**Why we care.** 
The result that matters is the following.

```{admonition} Any stationary point of a convex function is a global minimum
:class: tip
Let $f$ be convex and differentiable.  Then any $\bm{x}^{*}$ satisfying
$\nabla f(\bm{x}^{*})=\bm{0}$ minimises $f$ globally.
```

The proof is one line from Eq. (4.2): setting
$\bm{x}=\bm{x}^{*}$ makes the gradient term vanish, leaving
$f(\bm{y})\ge f(\bm{x}^{*})$ for every $\bm{y}$.  For a convex cost function,
therefore, we need only find a point where the gradient vanishes and we are
done -- there are no local minima to be trapped in and no saddle points to be
delayed by.

We have already used this twice.  In Section *The Hessian matrix* we showed that
the least-squares Hessian is $\bm{H}=\bm{X}^{T}\bm{X}$, positive semi-definite
because $\bm{z}^{T}\bm{X}^{T}\bm{X}\bm{z}=\|\bm{X}\bm{z}\|_2^{2}\ge0$, so the
OLS problem is convex and the normal equations (1.39)
deliver the global minimum.  Adding the Ridge penalty makes the Hessian
$\bm{X}^{T}\bm{X}+\lambda\bm{I}$, positive definite for $\lambda>0$, so the
Ridge problem is *strictly* convex and its minimum is unique.  The Lasso
cost of Eq. (3.41) is convex but not differentiable, which
is why Section *The Lasso* needed coordinate descent rather than a
gradient.

The bad news is that the cost functions of the later chapters are not convex.
A neural network with even one hidden layer has a cost surface with many local
minima, and the guarantee above does not apply.  We shall nonetheless use
methods designed for the convex case, because they work in practice; but it is
important to know that when we do so we are relying on empirical good
behaviour rather than on a theorem.


## Newton's method

Before descending gradients it is worth recalling the method that uses
curvature properly, both because it is the standard against which everything
else is measured and because its cost is what motivates the alternatives.

**Root finding in one dimension.** 
Newton's method, also called Newton-Raphson, finds a root of $f$ by extending
the tangent line at the current point until it crosses zero.  Taylor expanding
about a point $x$ close to the solution $s$,

$$
f(s) = 0 = f(x) + (s-x)f'(x) + \frac{(s-x)^{2}}{2}f''(x)+\dots,\tag{4.3}
$$

and discarding terms beyond the linear one gives $f(x)+(s-x)f'(x)\approx0$,
that is $s\approx x-f(x)/f'(x)$.  As an iteration,

$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)} .\tag{4.4}
$$

Geometrically $x_{n+1}$ is where the tangent at $(x_n,f(x_n))$ meets the
horizontal axis.  Close to a root the convergence is quadratic and extremely
fast.  Far from one, where the discarded terms matter, the formula can be
grossly inaccurate: if an iterate lands near a local extremum, so that
$f'$ nearly vanishes, the method can fail completely.  If the derivative is
available only numerically, or the function is not smooth, Newton's method is
best avoided.

**Several variables.** 
For a system $f_1(x_1,x_2)=0$, $f_2(x_1,x_2)=0$, Taylor expansion gives

$$
0 = f_k(x_1+h_1,x_2+h_2)
    = f_k(x_1,x_2) + h_1\frac{\partial f_k}{\partial x_1}
                   + h_2\frac{\partial f_k}{\partial x_2} + \dots,
  \qquad k=1,2,\tag{4.5}
$$

and collecting the partial derivatives into the Jacobian
matrix (1.16),

$$
\bm{J} = \begin{pmatrix}
    \partial f_1/\partial x_1 & \partial f_1/\partial x_2\\
    \partial f_2/\partial x_1 & \partial f_2/\partial x_2
  \end{pmatrix},\tag{4.6}
$$

the update becomes $\bm{x}^{n+1}=\bm{x}^{n}+\bm{h}^{n}$ with

$$
\bm{h}^{n} = -\bm{J}^{-1}\bm{f}(\bm{x}^{n}).\tag{4.7}
$$

We must invert the Jacobian, and difficulties arise when it is nearly
singular -- the conditioning problem of Section *Vector and matrix norms* again.

**Newton's method for minimisation.** 
Minimising $C(\bm{\theta})$ means finding a root of $\nabla C$, so we apply the
above with $\bm{f}=\nabla C$.  The Jacobian of the gradient is the Hessian,
and the iteration reads

$$
\boxed{\;
  \bm{\theta}_{k+1} = \bm{\theta}_k
    - \bm{H}^{-1}(\bm{\theta}_k)\,\nabla C(\bm{\theta}_k) . \;}\tag{4.8}
$$

Equation (4.8) is the ideal against which the rest of this
chapter should be read.  It is what one gets by minimising the second-order
Taylor expansion of the cost exactly, and it has two properties no
gradient-only method can match: it converges quadratically near the minimum,
and it is *invariant* under linear rescaling of the parameters, so that
the conditioning problems which dominate everything below simply do not arise.

Applied to ordinary least squares it is not merely fast but exact.  With
$\nabla C=-2\bm{X}^{T}(\bm{y}-\bm{X}\bm{\theta})/n$ from
Eq. (1.38) and $\bm{H}=2\bm{X}^{T}\bm{X}/n$ from
Eq. (1.44), a single step from any starting point gives

$$
\bm{\theta}_{1} = \bm{\theta}_0
    + \left(\bm{X}^{T}\bm{X}\right)^{-1}\bm{X}^{T}
      \left(\bm{y}-\bm{X}\bm{\theta}_0\right)
    = \left(\bm{X}^{T}\bm{X}\right)^{-1}\bm{X}^{T}\bm{y},\tag{4.9}
$$

the exact solution (3.8).  This is not a coincidence: for a
quadratic cost the second-order Taylor expansion is the function itself, so
one Newton step lands on the minimum.

**Why we do not simply use it.** 
For $p$ parameters the Hessian has $p^{2}$ entries and inverting it costs
$\bigO(p^{3})$ operations by Section *LU and Cholesky decompositions*.  For a linear regression
with twenty features this is nothing.  For a neural network with $10^{7}$
parameters the Hessian would need $10^{14}$ numbers, which cannot be stored,
let alone factorised.  The entire subject of this chapter is the construction
of methods that capture some of the benefit of Eq. (4.8) at a
cost linear rather than cubic in $p$.


## Gradient descent

The basic idea is that a function $F(\bm{x})$ decreases fastest, locally, if
one moves from $\bm{x}$ in the direction of the negative gradient
$-\nabla F(\bm{x})$.  One can show that for

$$
\bm{x}_{k+1} = \bm{x}_k - \gamma_k\nabla F(\bm{x}_k),
  \qquad \gamma_k>0,\tag{4.10}
$$

and for $\gamma_k$ small enough, $F(\bm{x}_{k+1})\le F(\bm{x}_k)$: we always
move towards smaller function values.  Iterating Eq. (4.10) from an
initial guess $\bm{x}_0$ is the method of *steepest descent*, or simply
*gradient descent* (GD).  The parameter $\gamma_k$ is the step length, or
in machine learning the *learning rate*, and we shall usually call it
$\eta$.

Ideally the sequence converges to a global minimum.  In general we do not know
whether a minimum we reach is global or local; but by
Section *Convexity*, if $F$ is convex the question does not arise.  The
scheme is conceptually simple and easy to implement.  It also has severe
limitations.

**Limitations.** 
In machine learning we are often faced with non-convex, high-dimensional cost
functions with many local minima.  Gradient descent is deterministic, so
unless the initial guess is good it will descend into whichever local minimum
it happens to face, and the result is sensitive to that initial condition.
The gradient is a function of all $p$ parameters and can be expensive to
evaluate.  And the method is delicate in its dependence on $\eta$: we are
guaranteed $F(\bm{x}_{k+1})\le F(\bm{x}_k)$ only for sufficiently small steps,
so too large a value produces erratic behaviour or divergence while too small
a value converges intolerably slowly.  Determining a good $\eta$ is the
central practical difficulty, and Sections *AdaGrad* to
*Adam* are devoted to removing the need to do so by hand.

Many of these shortcomings can be alleviated by introducing randomness, which
is the subject of Section *Stochastic gradient descent*.


## Gradient descent for linear regression

Linear regression is the ideal test case for gradient descent, for three
reasons: it has an analytical solution to check against, its gradient can be
computed exactly, and its cost function is convex so that convergence is
guaranteed for small enough learning rates.

We take the data set used throughout Chapter 3,

$$
y_i = 4 + 3x_i + \varepsilon_i,
  \qquad x_i \sim \mathcal{U}(0,2),
  \qquad \varepsilon_i\sim\mathcal{N}(0,1),
$$

with $n=100$ points, and the design matrix including an intercept column,

$$
\bm{X} = \begin{bmatrix}
    1 & x_0\\ \vdots & \vdots\\ 1 & x_{n-1}
  \end{bmatrix}
  \in\mathbb{R}^{n\times2},
  \qquad
  \bm{\theta}=\begin{pmatrix}\theta_0\\\theta_1\end{pmatrix}.\tag{4.11}
$$

The cost function is

$$
C(\bm{\theta}) = \frac{1}{n}\left\|\bm{X}\bm{\theta}-\bm{y}\right\|_2^{2}
   = \frac{1}{n}\sum_{i=0}^{n-1}
     \left[(\theta_0+\theta_1 x_i)^{2}
       -2y_i(\theta_0+\theta_1x_i)+y_i^{2}\right].\tag{4.12}
$$

Computing $\partial C/\partial\theta_0$ and $\partial C/\partial\theta_1$
separately and reassembling,

$$
\nabla_{\bm{\theta}}C(\bm{\theta})
   = \frac{2}{n}\begin{bmatrix}
       \sum_i\left(\theta_0+\theta_1x_i-y_i\right)\\
       \sum_i\left(x_i(\theta_0+\theta_1x_i)-y_ix_i\right)
     \end{bmatrix}
   = \frac{2}{n}\bm{X}^{T}\left(\bm{X}\bm{\theta}-\bm{y}\right),\tag{4.13}
$$

which is Eq. (1.38) written out coordinate by coordinate.
The Hessian is

$$
\bm{H} = \begin{bmatrix}
    \dfrac{\partial^{2}C}{\partial\theta_0^{2}} &
    \dfrac{\partial^{2}C}{\partial\theta_0\partial\theta_1}\\[8pt]
    \dfrac{\partial^{2}C}{\partial\theta_0\partial\theta_1} &
    \dfrac{\partial^{2}C}{\partial\theta_1^{2}}
  \end{bmatrix}
   = \frac{2}{n}\bm{X}^{T}\bm{X},\tag{4.14}
$$

in agreement with Eq. (1.44), and positive semi-definite, so
$C$ is convex and any stationary point is the global minimum.

The gradient descent iteration is

$$
\bm{\theta}_{k+1} = \bm{\theta}_k - \eta\,\nabla_{\bm{\theta}}C(\bm{\theta}_k),
  \qquad k=0,1,\dots\tag{4.15}
$$

started from a random $\bm{\theta}_0$ and stopped when
$\|\nabla_{\bm{\theta}}C\|\le\epsilon$ for some tolerance, say $10^{-8}$.


In [ ]:
import numpy as np

n = 100
rng = np.random.default_rng(2024)
x = 2.0 * rng.random((n, 1))
y = 4.0 + 3.0 * x + rng.normal(size=(n, 1))
X = np.c_[np.ones((n, 1)), x]

# The analytical solution of Chapter 3, for comparison
theta_exact = np.linalg.pinv(X.T @ X) @ X.T @ y
print("analytical:", theta_exact.ravel())

# Gradient descent, Eq. (4.gditeration)
theta = rng.normal(size=(2, 1))
eta = 0.1
for k in range(1000):
    gradient = (2.0 / n) * X.T @ (X @ theta - y)
    theta -= eta * gradient
    if np.linalg.norm(gradient) < 1.0e-8:
        break
print(f"gradient descent after {k+1} iterations:", theta.ravel())


With $\eta=0.1$ the iteration reproduces the analytical answer to five decimal
places within a few hundred steps.  With $\eta=0.001$, the value used in the
lecture notes, a thousand iterations are not nearly enough; with
$\eta$ too large it diverges.  The next section explains exactly where the
boundary lies.

### Gradient descent for Ridge regression

The same treatment applies to the Ridge cost of Eq. (3.27).
Writing it as

$$
C(\bm{\theta}) = \frac{1}{n}\left\|\bm{X}\bm{\theta}-\bm{y}\right\|_2^{2}
                 + \lambda\bm{\theta}^{T}\bm{\theta},\tag{4.16}
$$

the gradient and Hessian follow from
Eqs. (1.25) and (1.28),

$$
\nabla_{\bm{\theta}}C
   = \frac{2}{n}\bm{X}^{T}\left(\bm{X}\bm{\theta}-\bm{y}\right)
     + 2\lambda\bm{\theta},
  \qquad
  \bm{H} = \frac{2}{n}\bm{X}^{T}\bm{X} + 2\lambda\bm{I} .\tag{4.17}
$$

Setting the gradient to zero returns the closed
form (3.28), as it must.  The Hessian is now positive
definite for every $\lambda>0$, so the problem is strictly convex.


In [ ]:
import numpy as np

lmbda = 0.001
theta = rng.normal(size=(2, 1))
eta = 0.1
for k in range(1000):
    gradient = 2.0 * (X.T @ (X @ theta - y) / n + lmbda * theta)
    theta -= eta * gradient

# Compare with the closed form of Chapter 3
I = np.eye(X.shape[1])
theta_exact = np.linalg.inv(X.T @ X + n * lmbda * I) @ X.T @ y
print("gradient descent:", theta.ravel())
print("closed form:     ", theta_exact.ravel())


Note the factor $n$ in the closed form.  Our cost (4.16)
carries $1/n$ on the data term but not on the penalty, whereas
Eq. (3.28) was derived with neither; the two agree only if
$\lambda$ is scaled accordingly.  This is precisely the class of bookkeeping
error warned against in Section *Scaling, centring and the intercept*, and it is worth
tracking carefully whenever a penalised gradient is implemented.


## The learning rate and the condition number

We can now answer the question of how large $\eta$ may be, and in doing so
identify the quantity that governs everything in the rest of this chapter.

Consider a quadratic cost with symmetric positive definite Hessian $\bm{H}$,
which by Eq. (4.14) is the case for least squares.  Write the
error at step $k$ as $\bm{e}_k=\bm{\theta}_k-\bm{\theta}^{*}$, with
$\bm{\theta}^{*}$ the minimum.  Since $\nabla C(\bm{\theta})=\bm{H}\bm{e}$ for
a quadratic, the iteration (4.15) gives

$$
\bm{e}_{k+1} = \bm{e}_k - \eta\bm{H}\bm{e}_k
               = \left(\bm{I}-\eta\bm{H}\right)\bm{e}_k .\tag{4.18}
$$

Now diagonalise.  By the spectral decomposition (1.54),
$\bm{H}=\bm{Q}\bm{\Lambda}\bm{Q}^{T}$ with orthonormal eigenvectors and
eigenvalues $\lambda_i>0$.  Expressing the error in the eigenbasis,
$\tilde{\bm{e}}=\bm{Q}^{T}\bm{e}$, the recursion *decouples completely*:

$$
\tilde{e}_{k+1,i} = \left(1-\eta\lambda_i\right)\tilde{e}_{k,i},
  \qquad\text{so}\qquad
  \tilde{e}_{k,i} = \left(1-\eta\lambda_i\right)^{k}\tilde{e}_{0,i} .\tag{4.19}
$$

Each eigendirection contracts by its own factor $|1-\eta\lambda_i|$ at every
step, and this single equation contains everything.

**Stability.** 
The iteration converges along direction $i$ only if
$|1-\eta\lambda_i|<1$, that is $0<\eta<2/\lambda_i$.  For *all*
directions to converge we need

$$
\boxed{\;0 < \eta < \frac{2}{\lambda_{\max}} \;}\tag{4.20}
$$

with $\lambda_{\max}$ the largest eigenvalue of the Hessian.  Exceed it and
the component along the steepest direction grows geometrically: this is the
divergence one observes when the learning rate is set too high.  The bound is
set by the *steepest* direction, whatever the others are doing.

**Speed.** 
The slowest direction is the one with the smallest eigenvalue
$\lambda_{\min}$, contracting by $|1-\eta\lambda_{\min}|$ per step.  Choosing
$\eta$ to make the worst contraction factor as small as possible gives the
optimal value $\eta^{*}=2/(\lambda_{\max}+\lambda_{\min})$ and a contraction
rate

$$
\left|\frac{\lambda_{\max}-\lambda_{\min}}{\lambda_{\max}+\lambda_{\min}}\right|
   = \frac{\kappa-1}{\kappa+1},
  \qquad
  \kappa = \frac{\lambda_{\max}}{\lambda_{\min}} = \kappa_2(\bm{H}),\tag{4.21}
$$

where $\kappa$ is the condition number of the Hessian in the sense of
Eq. (1.77).  The number of iterations needed to reduce the error
by a fixed factor is therefore proportional to $\kappa$.

Figure 4.1 confirms both statements numerically.  The
iteration count falls as $\eta$ increases, reaches its minimum at the predicted
$\eta^{*}=2/(\lambda_{\max}+\lambda_{\min})$, rises again, and diverges
exactly at $2/\lambda_{\max}$.  There is no margin of safety at the right-hand
edge: the transition from the fastest convergence available to outright
divergence occupies a factor of two in $\eta$.

![Iterations required to reach an error of 10-6 against the learning rat](../BookML/BookFigures/chapter04_optimization/learning_rate_bound.png)

*Figure 4.1: Iterations required to reach an error of $10^{-6}$ against the learning rate, for a quadratic with eigenvalues $\{0.05,1,5\}$.  The optimum lies at $\eta^{*}$ and the method diverges beyond $2/\lambda_{\max}$, as predicted by Eqs. (4.20) and (4.21).*

This is the central result of the chapter, and it deserves to be read slowly.
Gradient descent is not slow because gradients are a bad idea.  It is slow
because it applies *the same* $\eta$ to every eigendirection, while
stability forces that single $\eta$ to be dictated by the steepest direction.
In a problem where $\lambda_{\max}/\lambda_{\min}=1000$, the step that is
barely stable along the steep direction is a thousand times too small along
the flat one, and the flat direction is where the remaining error lives.  The
iterates oscillate across the narrow valley while creeping along its floor.

**What this means for regression.** 
For least squares $\bm{H}=2\bm{X}^{T}\bm{X}/n$, so by
Eq. (1.118)

$$
\kappa(\bm{H}) = \kappa_2\left(\bm{X}^{T}\bm{X}\right)
                 = \kappa_2(\bm{X})^{2} ,\tag{4.22}
$$

and the iteration count scales with the *square* of the condition number
of the design matrix.  Three consequences follow, each connecting to earlier
chapters.

First, standardising the features is not only the statistical convention of
Section *Scaling, centring and the intercept* but a direct accelerator: it makes the
columns of $\bm{X}$ comparable in scale, reduces $\kappa$, and thereby reduces
the number of iterations.  This is the precise sense in which
"transform your inputs" is good advice.

Second, Ridge regression improves conditioning as well as variance.  By
Eq. (4.17) the Hessian becomes
$2(\bm{X}^{T}\bm{X}/n+\lambda\bm{I})$, whose condition number is
$(\lambda_{\max}+\lambda)/(\lambda_{\min}+\lambda)$ -- smaller than $\kappa$
for every $\lambda>0$.  Regularisation makes the optimisation easier as well
as the estimator better behaved, exactly as anticipated in the notebox of
Section *Vector and matrix norms*.

Third, the conjugate gradient method of Section *The conjugate gradient method* achieves a rate
governed by $\sqrt{\kappa}$ rather than $\kappa$, and for a quadratic cost it
is strictly superior to gradient descent.  It is the right tool for linear
regression at scale.  Its disadvantage is that it relies on the cost being
quadratic and on exact gradients, and it copes badly with the stochastic,
non-convex problems of the following chapters -- which is why the machine
learning literature developed a different set of remedies, to which we now
turn.

Figure 4.2 shows what the algebra describes.  On an elongated
quadratic with $\kappa=12$, a small learning rate creeps along the valley floor
without ever crossing it; a learning rate near the stability
bound (4.20) oscillates across the valley while making slow
progress along it; and momentum, at the same learning rate, damps the
oscillation and accelerates the drift, arriving in a fraction of the steps.
The three panels are the whole of this section and the next in pictures.

![Gradient descent on a quadratic with kappa12.  Left too small a step. ](../BookML/BookFigures/chapter04_optimization/gd_paths_conditioning.png)

*Figure 4.2: Gradient descent on a quadratic with $\kappa=12$.  Left: too small a step.  Centre: a step near the stability bound $2/\lambda_{\max}$, oscillating across the narrow direction.  Right: the same step with momentum $\gamma=0.85$, which cancels the oscillation and accumulates along the flat direction.*


## Momentum

The first remedy is to give the iteration a memory of where it has been.
Momentum-based gradient descent replaces Eq. (4.10) by

$$
\bm{v}_{k+1} = \gamma\bm{v}_k + \eta\nabla C(\bm{\theta}_k),
  \qquad
  \bm{\theta}_{k+1} = \bm{\theta}_k - \bm{v}_{k+1},\tag{4.23}
$$

with $\gamma\in[0,1)$ the *momentum parameter*, typically $0.9$.  The
update is no longer the current gradient but an exponentially weighted sum of
all gradients seen so far,

$$
\bm{v}_{k+1} = \eta\sum_{j=0}^{k}\gamma^{\,k-j}\nabla C(\bm{\theta}_j),\tag{4.24}
$$

with older gradients discounted geometrically.  Setting $\gamma=0$ recovers
plain gradient descent.

**Why it works.** 
Return to the decoupled recursion (4.19).  Along an
eigendirection with eigenvalue $\lambda_i$, the gradient is
$\lambda_i\tilde{e}_i$ and the momentum iteration becomes a two-term
recurrence in $\tilde{e}$ rather than a one-term one.  Its behaviour splits
into two regimes, and the split is exactly what we want.

In a *flat* direction, $\eta\lambda_i\ll1$, successive gradients point
the same way and Eq. (4.24) adds them up.  A constant
gradient $g$ accumulates to

$$
v_\infty = \eta g \sum_{j=0}^{\infty}\gamma^{\,j} = \frac{\eta g}{1-\gamma},\tag{4.25}
$$

so the effective step is multiplied by $1/(1-\gamma)$ -- a factor of ten for
$\gamma=0.9$.  The method accelerates precisely where plain gradient descent
crawls.

In a *steep* direction, the iterate overshoots and the gradient reverses
sign at every step.  Successive terms in Eq. (4.24) then
alternate in sign and largely cancel, so the accumulated step is
*smaller* than the bare gradient.  The oscillation across the narrow
valley is damped.

Momentum thus amplifies the consistent and cancels the oscillatory, which is
the right treatment for the ill-conditioned landscape diagnosed in
Section *The learning rate and the condition number*.  A careful analysis of the two-term recurrence
shows that with optimally chosen $\eta$ and $\gamma$ the contraction rate
improves from Eq. (4.21) to

$$
\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1},\tag{4.26}
$$

so the iteration count scales with $\sqrt{\kappa}$ instead of $\kappa$.  For
$\kappa=10^{4}$ that is a hundredfold reduction.  The reader will notice that
$\sqrt{\kappa}$ is also the rate of the conjugate gradient method quoted in
Section *The conjugate gradient method*; this is not a coincidence, since both build their step
out of the current gradient and the previous direction.  Momentum is,
in effect, conjugate gradients with the optimal coefficients replaced by a
fixed constant that need not be recomputed and does not require the cost to be
quadratic.

**A minimal example.** 
The behaviour is easiest to see on $f(x)=x^{2}$, where the exact answer is
known.

Figure 4.3 shows what the improvement from $\kappa$ to
$\sqrt{\kappa}$ is worth.  At $\kappa=10^{2}$ momentum saves an order of
magnitude in iterations; at $\kappa=10^{4}$, two.  Since
Eq. (4.22) tells us that the least-squares Hessian has the
condition number of the design matrix *squared*, condition numbers in
this range are entirely ordinary, and the saving is not academic.

![Iterations to fixed accuracy against the condition number, for gradien](../BookML/BookFigures/chapter04_optimization/momentum_rate.png)

*Figure 4.3: Iterations to fixed accuracy against the condition number, for gradient descent and for momentum with optimal parameters, from the rates (4.21) and (4.26).*


In [ ]:
import numpy as np

def objective(x):
    return x**2.0

def derivative(x):
    return 2.0 * x

def gradient_descent(derivative, bounds, n_iter, step_size, momentum=0.0,
                     rng=None):
    """Gradient descent with optional momentum, Eq. (4.momentum)."""
    rng = np.random.default_rng() if rng is None else rng
    solution = bounds[:, 0] + rng.random(len(bounds)) * (bounds[:, 1] - bounds[:, 0])
    change = 0.0
    solutions, scores = [], []
    for i in range(n_iter):
        gradient = derivative(solution)
        new_change = step_size * gradient + momentum * change
        solution = solution - new_change
        change = new_change
        solutions.append(solution.copy())
        scores.append(objective(solution))
    return solutions, scores


bounds = np.asarray([[-1.0, 1.0]])
rng = np.random.default_rng(4)
plain = gradient_descent(derivative, bounds, 30, 0.1, momentum=0.0,
                         rng=np.random.default_rng(4))
withmom = gradient_descent(derivative, bounds, 30, 0.1, momentum=0.3,
                           rng=np.random.default_rng(4))
print(f"after 30 steps: plain f = {plain[1][-1][0]:.3e}, "
      f"momentum f = {withmom[1][-1][0]:.3e}")


For this one-dimensional problem $\kappa=1$ and there is nothing to fix, yet
momentum still helps by increasing the effective step through
Eq. (4.25).  The dramatic gains appear when the eigenvalues
differ, which is the case for every realistic problem.

```{admonition} Machine learning connection
:class: tip
A widely used variant is
*Nesterov* momentum, which evaluates the gradient not at
$\bm{\theta}_k$ but at the point the momentum term is about to carry it to,
$\bm{\theta}_k-\gamma\bm{v}_k$.  The intuition is that of looking ahead before
stepping: if the accumulated velocity is about to overshoot, the gradient at
the extrapolated point already points backwards and corrects the step early.
For convex problems this improves the constant in
Eq. (4.26), and in deep learning frameworks it is usually
available as a flag on the standard momentum optimiser.
```


## Stochastic gradient descent

The second remedy attacks a different problem: cost.

**The idea.** 
Almost every cost function in machine learning is a sum over data points,

$$
C(\bm{\theta}) = \sum_{i=1}^{n} c_i(\bm{x}_i,\bm{\theta}),
  \qquad
  \nabla_{\bm{\theta}}C(\bm{\theta})
   = \sum_{i=1}^{n}\nabla_{\bm{\theta}}c_i(\bm{x}_i,\bm{\theta}) .\tag{4.27}
$$

Computing the full gradient therefore costs one pass over the entire data set
for a single parameter update.  In large-scale applications -- the ImageNet
challenge has millions of training images -- this is extravagant: we spend an
enormous computation to obtain one number, most of whose precision is wasted,
since we are going to take another step immediately afterwards.

Stochasticity is introduced by evaluating the gradient on a random subset of
the data, called a *minibatch*.  With $n$ data points and minibatches of
size $M$ there are $n/M$ minibatches, denoted $B_k$ with $k=1,\dots,n/M$.  If
$n=10$ and we choose five minibatches, each contains two points:
$B_1=(\bm{x}_1,\bm{x}_2)$ through $B_5=(\bm{x}_9,\bm{x}_{10})$.  Taking $M=n$
gives a single batch containing everything, which is ordinary gradient
descent; taking $M=1$ gives one point per batch.  We approximate

$$
\nabla_{\bm{\theta}}C(\bm{\theta})
   = \sum_{i=1}^{n}\nabla_{\bm{\theta}}c_i(\bm{x}_i,\bm{\theta})
   \;\longrightarrow\;
   \sum_{i\in B_k}\nabla_{\bm{\theta}}c_i(\bm{x}_i,\bm{\theta}),\tag{4.28}
$$

and the update becomes

$$
\bm{\theta}_{j+1} = \bm{\theta}_j
    - \eta_j\sum_{i\in B_k}\nabla_{\bm{\theta}}c_i(\bm{x}_i,\bm{\theta}),\tag{4.29}
$$

with $k$ drawn at random with equal probability from $[1,n/M]$.  One pass
through all the minibatches is called an *epoch*, and one typically
chooses a number of epochs and iterates over the minibatches within each.

Strictly, *stochastic gradient descent* means using a single example at a
time, and the minibatch version should be called minibatch gradient descent;
but in practice everyone says SGD and means minibatches, and we shall do the
same.  Single-example updates are in fact rarely used, because vectorised
hardware evaluates a gradient on $100$ examples far faster than $100$
single-example gradients.  The minibatch size is a hyperparameter, but it is
seldom tuned by cross-validation: it is usually set by memory constraints, or
to a power of two such as $32$, $64$ or $128$, because vectorised operations
are fastest at those sizes.


In [ ]:
import numpy as np

n = 100            # data points
M = 5              # size of each minibatch
m = int(n / M)     # number of minibatches
n_epochs = 10

for epoch in range(1, n_epochs + 1):
    for i in range(m):
        k = np.random.randint(m)     # pick the k-th minibatch at random
        # compute the gradient using the data in minibatch B_k
        # update theta with Eq. (4.sgdupdate)
        pass


**Two benefits.** 
Taking the gradient on a subset has two distinct advantages.  It is
*cheaper*: if $M\ll n$ the gradient costs a fraction of the full one, so
we take many more steps for the same computation.  And it introduces
*noise*, which decreases the chance of the scheme becoming stuck in a
poor local minimum -- the deterministic objection raised in
Section *Gradient descent*.  A stochastic iterate can be jostled out of a
shallow basin in a way a deterministic one cannot.

The corresponding disadvantages are that convergence is erratic rather than
monotone, that the iterate does not settle at the minimum but rattles around
in a neighbourhood whose size is set by the gradient noise and the learning
rate, and that the learning rate must be chosen with more care.

**Speed against accuracy.** 
It is worth being precise about the trade-off, because the folk statement that
"SGD converges faster" conflates two different things.  Per *iteration*,
full-batch gradient descent makes more progress: it uses the exact gradient
and, for a strongly convex problem, converges geometrically at the rate of
Eq. (4.21).  Stochastic gradient descent, with a decaying
learning rate, converges at the far slower rate $\bigO(1/k)$ for strongly
convex problems and $\bigO(1/\sqrt{k})$ in general, because the gradient noise
does not vanish.  Per *unit of computation*, however, SGD is
overwhelmingly ahead, since each of its iterations costs $M/n$ of a full one.
For a data set of a million points and a batch of a hundred, SGD performs ten
thousand updates in the time full-batch gradient descent performs one.

The memory picture is similarly stark: full-batch methods must hold the whole
data set, or at least stream it, for every update, whereas SGD needs only the
current minibatch.  This is what makes training on data sets larger than
memory possible at all, and it is the reason every deep learning framework is
built around minibatches.

By Section *The central limit theorem*, the noise in a minibatch gradient estimate falls as
$1/\sqrt{M}$: quadrupling the batch size halves the gradient noise at four
times the cost per step.  That unfavourable exchange rate is why very large
batches are not automatically better.

### Learning rate schedules and stopping

Because the gradient noise does not decay, a constant learning rate leaves the
iterate bouncing around the minimum forever.  The standard remedy is to let
$\eta$ decay with time.  Let $e=0,1,2,\dots$ index the epoch, let $m$ be the
number of minibatches, and set $t=e\cdot m+i$ with $i=0,\dots,m-1$.  A common
choice is

$$
\eta_j(t;t_0,t_1) = \frac{t_0}{t+t_1},\tag{4.30}
$$

with $t_0,t_1>0$ fixed.  The learning rate starts at $t_0/t_1$ and decays
towards zero, so that the iterate eventually stops moving.  The classical
conditions for convergence, due to Robbins and Monro, are that
$\sum_t\eta_t=\infty$ -- so that the iterate can still reach the minimum from
any starting point -- while $\sum_t\eta_t^{2}<\infty$, so that the accumulated
noise is finite.  Equation (4.30) satisfies both.


In [ ]:
import numpy as np

def step_length(t, t0, t1):
    return t0 / (t + t1)

n, M = 100, 5
m = int(n / M)
n_epochs, t0, t1 = 500, 1.0, 10.0

for epoch in range(1, n_epochs + 1):
    for i in range(m):
        k = np.random.randint(m)
        t = epoch * m + i
        eta = step_length(t, t0, t1)
        # compute the minibatch gradient and update theta
print(f"eta after {n_epochs} epochs: {step_length(n_epochs*m, t0, t1):g}")


**When do we stop?.** 
One possibility is to compute the full gradient every few epochs and stop when
its norm falls below a threshold.  But a vanishing gradient identifies a
stationary point, not necessarily a good one, so it is wiser to evaluate the
cost at that point, record it, and continue: if the criterion triggers again
later, keep whichever $\bm{\theta}$ gave the lower value.  Because the scheme
is random by design, repeating the whole computation gives a different answer
each time, and taking the best of several runs is a legitimate and common
strategy.

In machine learning the more useful criterion is not about the gradient at
all.  As advised in Section *Training error, test error and generalisation*, one monitors the cost on
a held-out validation set and stops when it begins to *rise* while the
training cost is still falling.  This is *early stopping*, and it is a
regularisation method as much as a termination rule: halting before the
variance term of Eq. (2.47) has grown is another way of
trading a little bias for a lot of variance, in the same spirit as the Ridge
penalty of Section *Ridge regression*.


## Why adapt the step size at all

In stochastic gradient descent, with or without momentum, we still have to
specify a schedule for the learning rate $\eta_t$.  Section *The learning rate and the condition number*
showed exactly why this is awkward, and it is worth restating the diagnosis in
the form that motivates what follows.

A fixed $\eta$ is hard to get right.  If it is too large the updates overshoot
and the iteration oscillates or diverges, by Eq. (4.20); if it
is too small, convergence takes forever, by Eq. (4.21).  But the
deeper problem is not the choice of a number: it is that *no single
number is right for all directions*.  The stability bound is set by
$\lambda_{\max}$ and the convergence rate by $\lambda_{\min}$, and when these
differ by orders of magnitude the step which is barely safe along the steep
direction is hopelessly small along the flat one.  Steep coordinates need
small steps; flat coordinates could take large ones.  In high-dimensional
problems with features of varying scale, or with sparse features that are
active only occasionally, the mismatch is severe.

Ideally our algorithm would keep track of curvature and take large steps in
shallow directions and small steps in steep ones.  That is precisely what
Newton's method (4.8) does, by normalising the gradient with
$\bm{H}^{-1}$; and the invariance this buys is why
Section *Newton's method* called it the ideal.  The trouble, again, is the
$\bigO(p^{3})$ cost of forming and inverting the Hessian, which is out of the
question for large models.

The methods of this section are a compromise.  They approximate the curvature
information in $\bm{H}$ using only quantities already available -- the
gradients themselves -- and they restrict the approximation to a
*diagonal* matrix, so that inverting it costs $\bigO(p)$ rather than
$\bigO(p^{3})$.  Instead of tracking the gradient alone, they track the
*second moment* of the gradient.  The family includes AdaGrad, AdaDelta,
RMSProp and Adam, and we develop the three named in the chapter title.

**The idea in one paragraph.** 
Suppose we replace the scalar $\eta$ by a diagonal matrix, so that coordinate
$j$ has its own step size $\eta/\sqrt{r_j}$.  What should $r_j$ be?  Near a
minimum the gradient along direction $j$ behaves as $g_j\approx\lambda_j e_j$,
so a large curvature $\lambda_j$ produces large gradients and a small
curvature produces small ones.  *The typical magnitude of the gradient in
a direction is a proxy for the curvature in that direction.*  Averaging
$g_j^{2}$ over recent steps therefore estimates something like $\lambda_j^{2}$
without ever forming a second derivative, and dividing by its square root
approximates the $\bm{H}^{-1/2}$ scaling.  This is the entire principle behind
AdaGrad, RMSProp and Adam; the three differ only in how the average is taken.


## AdaGrad

AdaGrad maintains a running sum of squared gradients for each coordinate.  Let
$\bm{g}_t=\nabla C_{i_t}(\bm{\theta}_t)$ be the gradient at step $t$, possibly
from a minibatch, and initialise $\bm{r}_0=\bm{0}$.  At each iteration
accumulate

$$
\bm{r}_t = \bm{r}_{t-1} + \bm{g}_t\circ\bm{g}_t,\tag{4.31}
$$

where $\circ$ is the Hadamard product of Section *Vectors*, so that
$r_{t,j}=r_{t-1,j}+g_{t,j}^{2}$ for every coordinate $j$.  We may regard
$\bm{H}_t=\mathrm{diag}(\bm{r}_t)$ as a diagonal matrix of accumulated squared
gradients, with $\bm{H}_0=\bm{0}$.

The update scales the gradient by the inverse square root of this matrix,

$$
\bm{\theta}_{t+1} = \bm{\theta}_t - \eta\,\bm{H}_t^{-1/2}\bm{g}_t,\tag{4.32}
$$

where $\bm{H}_t^{-1/2}$ is diagonal with entries $r_{t,j}^{-1/2}$.  In
coordinates each parameter has its own step size,

$$
\theta_{t+1,j} = \theta_{t,j} - \frac{\eta}{\sqrt{r_{t,j}}}\,g_{t,j},\tag{4.33}
$$

and in practice a small constant $\epsilon$ is added to the denominator for
numerical stability,

$$
\boxed{\;
  \theta_{t+1,j} = \theta_{t,j}
    - \frac{\eta}{\sqrt{\epsilon + r_{t,j}}}\,g_{t,j} . \;}\tag{4.34}
$$

The effective learning rate for parameter $j$ at time $t$ is
$\alpha_{t,j}=\eta/\sqrt{\epsilon+r_{t,j}}$, which decreases as $r_{t,j}$
grows.

Note the resemblance between Eq. (4.32) and Newton's
step (4.8): both premultiply the gradient by an inverse
matrix built from curvature information.  AdaGrad differs in using
$\bm{H}^{-1/2}$ rather than $\bm{H}^{-1}$, in restricting $\bm{H}$ to be
diagonal, and in estimating it from gradient magnitudes rather than second
derivatives.

**Properties.** 
AdaGrad tunes the step size for each parameter automatically.  Parameters with
large or volatile gradients receive smaller steps; those with small or
infrequent gradients receive relatively larger ones.  No manual schedule is
needed: because $\bm{r}_t$ never decreases, the step sizes
$\eta/\sqrt{r_{t,j}}$ are non-increasing, which has an effect similar to a
decaying learning rate but individualised per coordinate.

The benefit for sparse data is worth spelling out, since it is the setting for
which AdaGrad was designed.  Consider a rare feature -- a word appearing in
one document in a thousand.  Its gradient is zero on almost every minibatch,
so $r_{t,j}$ grows very slowly, so its learning rate stays high.  When the
feature finally does appear, the parameter takes a large, useful step instead
of the negligible one a global schedule would have permitted by then.
Frequently active features, conversely, accumulate large $r_{t,j}$ and their
learning rates fall automatically.

In convex optimisation AdaGrad achieves a convergence rate comparable to the
best fixed learning rate tuned in hindsight for the problem, which is a strong
guarantee and effectively removes the need to tune $\eta$ by hand.

**The limitation.** 
Because $\bm{r}_t$ accumulates without bound, the learning rates decay
monotonically and can become vanishingly small long before the minimum is
reached.  On a convex problem this is tolerable, since the accumulated
gradients genuinely reflect the geometry.  In deep learning, where training
runs for many epochs and the landscape changes character as the iterate moves,
it is fatal: AdaGrad simply stops making progress.  The sum has an infinite
memory, and remembers gradients from a region of parameter space the iterate
left long ago.  RMSProp and Adam repair exactly this.


## RMSProp

RMSProp replaces the cumulative sum (4.31) by an
exponentially decaying average,

$$
\bm{v}_t = \rho\,\bm{v}_{t-1} + (1-\rho)\left(\nabla C(\bm{\theta}_t)\right)^{2},\tag{4.35}
$$

with the square taken element-wise and $\rho$ typically $0.9$ or $0.99$.  The
update is

$$
\boxed{\;
  \bm{\theta}_{t+1} = \bm{\theta}_t
    - \frac{\eta}{\sqrt{\bm{v}_t+\epsilon}}\,\nabla C(\bm{\theta}_t) , \;}\tag{4.36}
$$

with the division element-wise.  The method was proposed by Geoffrey Hinton in
lecture notes in 2012 and never formally published, which has not prevented it
from becoming standard.

**Why the change matters.** 
Expanding Eq. (4.35) shows what has happened,

$$
\bm{v}_t = (1-\rho)\sum_{j=0}^{t}\rho^{\,t-j}\bm{g}_j^{2},\tag{4.37}
$$

which is a weighted average with weights summing to nearly one, rather than an
unbounded sum.  Gradients older than roughly $1/(1-\rho)$ steps -- ten steps
for $\rho=0.9$, a hundred for $\rho=0.99$ -- are effectively forgotten.  Two
consequences follow.  The quantity $\bm{v}_t$ is now an estimate of the
*recent* mean square gradient, so it tracks the local curvature as the
iterate moves through a changing landscape.  And because it does not grow
without bound, the effective learning rate does not decay to zero: AdaGrad's
infinite memory problem is gone.

RMSProp is thus the same idea as AdaGrad with a finite memory, and the
resemblance to the momentum update (4.23) is not accidental.
Both are exponential moving averages; momentum averages the gradient, RMSProp
averages its square.  It is natural to ask what happens if one does both.


## Adam

Adam -- adaptive moment estimation -- was introduced by Kingma and Ba in
2014 [kingma2014] and does exactly that.  It keeps running averages of
both the first and the second moment of the gradient and uses them to adapt
the learning rate for each parameter.  It is efficient for large problems
involving many data and many parameters, and it is the default optimiser in
most deep learning frameworks.

**Why combine momentum and RMSProp?.** 
The two mechanisms address different defects and do not overlap.  Momentum
gives fast convergence by smoothing the gradient, accelerating along the
consistent long-term direction as in Eq. (4.25) and damping
oscillations across narrow valleys.  RMSProp gives per-dimension scaling for
stability, handling features of different scales and sparse gradients.  Using
both means the direction of the step is chosen by an averaged gradient while
its length in each coordinate is chosen by the recent curvature estimate.

**The two moments.** 
Adam maintains, at each step $t$,

$$
\begin{align}
\bm{m}_t &= \beta_1\bm{m}_{t-1} + (1-\beta_1)\nabla C(\bm{\theta}_t)
   && \text{(first moment, the momentum term)},
  \\
  \bm{v}_t &= \beta_2\bm{v}_{t-1}
    + (1-\beta_2)\left(\nabla C(\bm{\theta}_t)\right)^{2}
   && \text{(second moment, the RMS term)},
\end{align}
$$

with typical values $\beta_1=0.9$ and $\beta_2=0.999$, and
$\bm{m}_0=\bm{v}_0=\bm{0}$.

**Bias correction.** 
Those initialisations cause a problem which is worth deriving, because it
explains a step that otherwise looks arbitrary.  Suppose the gradient is
stationary with true second moment $\mathbb{E}[g^{2}]$.  Unrolling
Eq. (4.39) as in Eq. (4.37) and taking the
expectation,

$$
\mathbb{E}[v_t] = (1-\beta_2)\sum_{j=1}^{t}\beta_2^{\,t-j}\,\mathbb{E}[g^{2}]
   = \mathbb{E}[g^{2}]\left(1-\beta_2^{\,t}\right),\tag{4.40}
$$

using the finite geometric sum.  The estimate is therefore too small by
exactly the factor $1-\beta_2^{t}$, and the same argument applies to
$\bm{m}_t$ with $\beta_1$.  The bias is severe at the start: with
$\beta_2=0.999$ the factor is $10^{-3}$ at $t=1$, so the raw $v_1$ underestimates
the true second moment by three orders of magnitude, and the step
$\eta/\sqrt{v}$ would be enormous.  Dividing by the known factor removes the
bias exactly,

$$
\hat{\bm{m}}_t = \frac{\bm{m}_t}{1-\beta_1^{\,t}},
  \qquad
  \hat{\bm{v}}_t = \frac{\bm{v}_t}{1-\beta_2^{\,t}} .\tag{4.41}
$$

For small $t$ the correction is large, compensating for the initial zero; as
$t$ grows, $1-\beta_i^{t}\to1$ and the corrected moments converge to the raw
ones.  Bias correction is what makes Adam stable in its first iterations, and
it is the one ingredient AdaGrad and RMSProp lack.

**The update.** 
Finally,

$$
\boxed{\;
  \bm{\theta}_{t+1} = \bm{\theta}_t
    - \frac{\alpha}{\sqrt{\hat{\bm{v}}_t}+\epsilon}\,\hat{\bm{m}}_t , \;}\tag{4.42}
$$

with $\epsilon$ a small constant, typically $10^{-8}$, preventing division by
zero.  Step by step: compute the gradient; update the two moving
averages (4.38) and (4.39); bias-correct with
Eq. (4.41); form the step
$\Delta\bm{\theta}_t=\hat{\bm{m}}_t/(\sqrt{\hat{\bm{v}}_t}+\epsilon)$; and
update the parameters.

**Adam against its predecessors.** 
AdaGrad uses per-coordinate scaling like Adam but has no momentum, and slows
down excessively because its accumulation never forgets.  RMSProp uses a
moving average of squared gradients, so it does not slow down, but includes
neither momentum nor bias correction.  Adam is, in effect, RMSProp plus
momentum plus bias correction: the first moment provides acceleration and
smoother convergence, the second moderates the step size per dimension, and
the correction ensures the estimates are sound from the first iteration.

```{admonition} Why these methods work: the summary argument
:class: tip
It is worth collecting
the thread that runs through Sections *The learning rate and the condition number* to
*Adam*, because each method is a response to the same diagnosis.

The trouble with gradient descent is the mismatch between one global step size
and a cost function whose curvature varies by direction; the iteration count
scales with $\kappa=\lambda_{\max}/\lambda_{\min}$ by
Eq. (4.21).  Newton's method removes the mismatch entirely by
premultiplying with $\bm{H}^{-1}$, but costs $\bigO(p^{3})$.

Momentum attacks the symptom: it cancels the oscillation along steep
directions and accumulates progress along flat ones, improving the rate from
$\kappa$ to $\sqrt{\kappa}$ at no extra cost.  It does not, however, use
different step sizes in different coordinates.

The adaptive methods attack the cause, by constructing a cheap diagonal
approximation to the curvature from the second moment of the gradient.  Where
gradients are persistently large the curvature is presumed large and the step
is shortened; where they are small the step is lengthened.  Dividing by
$\sqrt{v_j}$ is an $\bigO(p)$ stand-in for the $\bigO(p^{3})$ operation of
applying $\bm{H}^{-1/2}$.

Two observations follow.  The first is that these methods are approximately
invariant to the scaling of individual features, since multiplying a feature
by $c$ multiplies its gradient by $c$ and its accumulated second moment by
$c^{2}$, leaving the ratio unchanged.  That is why they are so much less
sensitive to preprocessing than plain gradient descent -- although, as
Section *Practical tips* notes, standardising the inputs remains good
practice.  The second is that a diagonal approximation can only capture
curvature aligned with the coordinate axes.  A cost function whose narrow
valley runs diagonally in parameter space has a Hessian with large
off-diagonal entries, and no diagonal rescaling will fix it; this is the
residual gap between Adam and true second-order methods, and it is why
decorrelating the inputs helps even when the scales already match.
```


## Implementations

The three algorithms are described in detail in Goodfellow, Bengio and
Courville [goodfellow2016], chapter 8.  We give compact implementations
here, all applied to the same least-squares problem of
Section *Gradient descent for linear regression* so that they may be compared directly.


In [ ]:
import numpy as np

def make_batches(n, batch_size, rng):
    """Shuffle the indices and split them into minibatches."""
    idx = rng.permutation(n)
    return [idx[i:i + batch_size] for i in range(0, n, batch_size)]


def sgd_adaptive(X, y, method="adam", n_epochs=100, batch_size=10,
                 eta=0.01, gamma=0.9, rho=0.99,
                 beta1=0.9, beta2=0.999, eps=1e-8, rng=None):
    """Stochastic gradient descent with the optimisers of this chapter.

    method is one of "plain", "momentum", "adagrad", "rmsprop", "adam".
    """
    rng = np.random.default_rng() if rng is None else rng
    n, p = X.shape
    theta = rng.normal(size=(p, 1))

    change = np.zeros((p, 1))          # momentum velocity, Eq. (4.momentum)
    r = np.zeros((p, 1))               # accumulated second moment
    m = np.zeros((p, 1))               # first moment, Eq. (4.adamfirst)
    t = 0

    for epoch in range(n_epochs):
        for batch in make_batches(n, batch_size, rng):
            t += 1
            Xb, yb = X[batch], y[batch]
            g = (2.0 / len(batch)) * Xb.T @ (Xb @ theta - yb)

            if method == "plain":
                update = eta * g
            elif method == "momentum":
                change = eta * g + gamma * change
                update = change
            elif method == "adagrad":
                r += g * g                                   # Eq. (4.adagradaccum)
                update = eta * g / (np.sqrt(r) + eps)        # Eq. (4.adagrad)
            elif method == "rmsprop":
                r = rho * r + (1 - rho) * g * g              # Eq. (4.rmspropaccum)
                update = eta * g / (np.sqrt(r) + eps)        # Eq. (4.rmsprop)
            elif method == "adam":
                m = beta1 * m + (1 - beta1) * g              # Eq. (4.adamfirst)
                r = beta2 * r + (1 - beta2) * g * g          # Eq. (4.adamsecond)
                m_hat = m / (1 - beta1**t)                   # Eq. (4.adambias)
                r_hat = r / (1 - beta2**t)
                update = eta * m_hat / (np.sqrt(r_hat) + eps)
            else:
                raise ValueError(f"unknown method {method}")

            theta -= update

    return theta


Running all five on the data of Section *Gradient descent for linear regression* for a hundred
epochs with batches of ten, for three values of the learning rate, gives the
final cost values of Table 4.1.  The analytical solution of
Eq. (3.8) is $\hat{\bm{\theta}}=(4.030,2.806)$ with cost
$1.14947$, which is the floor.

| **Method** | $\eta=0.01$ | $\eta=0.1$ | $\eta=0.5$ |
|---|---|---|---|
| Plain SGD | $1.1499$ | $1.1532$ | $1.2274$ |
| Momentum | $1.1553$ | $1.5674$ | $1.3490$ |
| AdaGrad | $28.049$ | $1.2776$ | $\mathbf{1.1497}$ |
| RMSProp | $1.1516$ | $1.1512$ | $1.1904$ |
| Adam | $1.2185$ | $1.1514$ | $1.1565$ |

*Table 4.1: Final value of the cost function (4.12) after $100$
epochs of stochastic gradient descent on the data of
Section *Gradient descent for linear regression*, for each optimiser and three learning rates.  The
analytical minimum is $1.14947$.  Note that no row is best at the same $\eta$
as any other.*

The table repays study, and not because it flatters the adaptive methods.

The first observation is that on this problem the sophisticated methods are
not better.  With $\eta=0.01$ plain SGD reaches $1.1499$, within $0.04\%$ of
the optimum, while Adam manages only $1.2185$ and AdaGrad is catastrophic at
$28.05$ -- it has not arrived anywhere near the minimum.  This is exactly what
Section *AdaGrad* predicted: the accumulated $\bm{r}_t$ drives the
effective step $\eta/\sqrt{r_t}$ towards zero, and with $\eta$ already small
the iterate stalls long before reaching the solution.  Nothing is wrong with
the implementation; the method is behaving as designed, and the design is
wrong for this problem.

The second observation explains the first.  Look along the rows rather than
down the columns.  AdaGrad is worst at $\eta=0.01$ and *best of all five*
at $\eta=0.5$, where it essentially attains the analytical minimum.  Plain SGD
and momentum move the other way, degrading as $\eta$ grows.  The reason is
that these methods do not use $\eta$ to mean the same thing.  Plain gradient
descent multiplies $\eta$ by the raw gradient, so the stability
bound (4.20) applies directly.  The adaptive methods divide by
$\sqrt{v_j}$ first, which renormalises the gradient to something of order
unity, so their $\eta$ sets a step length in parameter space rather than a
multiple of the gradient.  *A learning rate is not comparable across
optimisers*, and a comparison at a single fixed $\eta$ -- which is what one
sees most often -- says more about which method happens to suit that number
than about the methods.

The third observation is the one to carry forward.  This problem has $p=2$,
is convex, and has a Hessian with eigenvalues $0.311$ and $3.728$, hence
$\kappa=11.97$ by Eq. (4.21).  That is a benign landscape, and
there is simply nothing for the adaptive machinery to repair; the overhead of
estimating curvature buys nothing because the curvature is already uniform.
The adaptive methods earn their place when $\kappa$ is large, when $p$ is
large, when the gradients are sparse, and when the cost is non-convex --
which is to say in the neural networks of the following chapters, and not
here.  Demonstrating a method on a problem it was not designed for is a good
way to understand what it actually does.

Figure 4.4 plots the same comparison as a function of the
epoch rather than as a final number, and it makes the second lesson of the
table unmistakable.  At $\eta=0.01$ the adaptive methods are slower than plain
stochastic gradient descent and AdaGrad has effectively stopped; at $\eta=0.5$
the ordering is reversed and AdaGrad is the best of the five.  Nothing about
the methods has changed between the panels.  Only the number $\eta$ has, and it
means something different to each of them.

![Excess cost against epoch for the five optimisers of Section Implement](../BookML/BookFigures/chapter04_optimization/optimiser_comparison.png)

*Figure 4.4: Excess cost against epoch for the five optimisers of Section *Implementations* at two learning rates.  The vertical axis is $C(\bm{\theta})$ minus its value at the analytical minimum.  The ranking of the methods reverses between the panels.*

```{admonition} Machine learning connection
:class: tip
The default hyperparameters
$\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$ and $\alpha=10^{-3}$ from
the original paper are used almost universally and are usually a reasonable
starting point, which is a large part of Adam's popularity.  It should be
said, however, that adaptive methods do not always generalise as well as
plain SGD.  Several studies have found that models trained with Adam, RMSProp
or AdaGrad reach a worse test error than the same models trained with
well-tuned SGD with momentum, particularly in the overparameterised regime
where the number of parameters exceeds the number of data points.  Why this
should be so is not settled.  The practical advice is that Adam is an
excellent default for getting a model training at all, and that a carefully
tuned SGD with momentum is worth trying before the final result is reported.
```


## None of these can compete with Newton's method

It is a useful corrective, having built up this machinery, to see how it fares
against Eq. (4.8) on a problem where the Hessian is
affordable.  For the least-squares cost of Section *Gradient descent for linear regression* a single
Newton step lands on the exact minimum, by Eq. (4.9), while
the methods above need hundreds or thousands of iterations to get five decimal
places.


In [ ]:
import numpy as np

# One Newton step solves the least-squares problem exactly
H = (2.0 / n) * X.T @ X
theta = rng.normal(size=(2, 1))
gradient = (2.0 / n) * X.T @ (X @ theta - y)
theta -= np.linalg.solve(H, gradient)
print("after one Newton step:", theta.ravel())
print("analytical solution:  ", (np.linalg.pinv(X.T @ X) @ X.T @ y).ravel())


The two agree to machine precision.  The moral is not that we should use
Newton's method -- for a quadratic cost we would use the closed form of
Chapter 3, and for a large model we could not afford the
Hessian at all -- but that the gradient methods of this chapter are
*approximations* to something better, adopted because of cost.  Every
improvement from momentum to Adam is a step back towards
Eq. (4.8), recovering a little more curvature information for
a little more bookkeeping, and the quality of a method can fairly be judged by
how much of the Hessian it manages to imitate for $\bigO(p)$ work per step.


## Automatic differentiation

Every method in this chapter needs a gradient.  For linear regression we
computed it by hand in Eq. (4.13); for a neural network with
a dozen layers, doing so by hand is possible but error-prone, and for a model
under active development it is impractical.  *Automatic differentiation*
(AD), also called algorithmic differentiation, solves the problem completely.

AD exploits the fact that every computer program, however complicated,
executes a sequence of elementary arithmetic operations and elementary
functions -- addition, multiplication, $\exp$, $\log$, $\sin$ -- each of whose
derivatives is known.  Applying the chain rule of Eq. (1.51)
repeatedly to that sequence yields derivatives of arbitrary order,
*accurate to working precision*, at a cost only a small constant factor
above that of the original program.

It is important to distinguish AD from the two things it is often confused
with.  It is not *symbolic* differentiation, which manipulates
expressions and can produce enormous formulae, and which requires the program
to be converted into a single closed-form expression in the first place.  Nor
is it *numerical* differentiation by finite differences, which suffers
both discretisation error and, worse, catastrophic cancellation when the step
is made small -- the loss of significant digits described in
Section *Vector and matrix norms*.  AD has neither defect: it is exact, up to rounding,
and it costs a constant factor rather than one extra function evaluation per
parameter.

That last point is what makes deep learning possible.  Reverse-mode AD, which
is what backpropagation is, computes the gradient of a scalar cost with
respect to *all* $p$ parameters at a cost of roughly two forward
evaluations, independent of $p$.  Finite differences would need $p+1$
evaluations.  For $p=10^{7}$ the difference is between a fraction of a second
and a week.  The reason, as noted after Eq. (1.50), is
that for a scalar output it is far cheaper to accumulate the chain rule from
the left, row times matrix, than from the right.

**Autograd.** 
The `autograd` package differentiates ordinary numpy code.


In [ ]:
import autograd.numpy as np
from autograd import grad

def f(x):
    return np.sin(2 * np.pi * x + x**2)

df = grad(f)                 # df is a Python function, the derivative of f
print(df(1.0))

# It composes: the second derivative is just grad applied twice
d2f = grad(grad(f))
print(d2f(1.0))


Applied to our least-squares problem, the gradient we derived by hand in
Eq. (4.13) need never be written down:


In [ ]:
import autograd.numpy as np
from autograd import grad

def cost(theta, X, y):
    return np.sum((X @ theta - y)**2) / len(y)

training_gradient = grad(cost, 0)      # differentiate w.r.t. argument 0

theta = np.random.randn(2, 1)
eta = 0.1
for k in range(1000):
    theta -= eta * training_gradient(theta, X, y)


The result agrees with the hand-coded version to machine precision, which is
worth checking once as a matter of habit: comparing an analytical gradient
against an automatically differentiated one, or against a finite difference,
is the standard way of catching an error in a derivation.

**JAX.** 
For current work we recommend *JAX* rather than `autograd`.  JAX
combines the same differentiation machinery with XLA, so that the resulting
code can additionally be just-in-time compiled, vectorised and run on GPUs.


In [ ]:
import jax.numpy as jnp
from jax import grad, jit, vmap

def sum_logistic(x):
    return jnp.sum(1.0 / (1.0 + jnp.exp(-x)))

x_small = jnp.arange(3.0)
derivative_fn = grad(sum_logistic)
print(derivative_fn(x_small))

fast_derivative = jit(derivative_fn)      # compiled on first call


The transformations compose: `jit(grad(f))` compiles the gradient,
`vmap(grad(f))` vectorises it over a batch, and `grad(grad(f))`
differentiates it again.  We shall use this machinery freely from Chapter 5 onwards.


## Practical tips

The following advice is collected from the sources cited in this chapter and
from experience; each item connects to something already derived.

**Randomise the data when forming minibatches.** 
Always shuffle before splitting into minibatches.  Otherwise the ordering of
the data becomes a systematic feature of the gradient sequence, and the method
can fit spurious correlations arising purely from the order of presentation.
If the data arrive sorted by class, an unshuffled minibatch may contain a
single class and its gradient will point somewhere unhelpful.

**Transform your inputs.** 
Learning is difficult when the landscape mixes steep and flat directions, and
Section *The learning rate and the condition number* made that statement quantitative: the iteration
count scales with $\kappa$.  Standardising the inputs -- subtracting the mean
and dividing by the standard deviation, as in Section *Arrays in practice: numpy, BLAS and LAPACK* --
equalises the scales and reduces $\kappa$.  Whenever possible, decorrelate the
inputs as well.  The reason is exactly Eq. (1.44): for a squared
error cost the Hessian *is* the correlation matrix of the inputs, up to
normalisation, so standardising and decorrelating make the landscape as close
to spherical as it can be made, which by Eq. (4.21) is the
best case.  Since most deep networks are linear transformations followed by
non-linearities, the intuition carries beyond the linear case; this is also
the motivation for batch normalisation.

**Monitor out-of-sample performance.** 
Always track the cost on a validation set held out of training, as a proxy for
the test set of Section *Training error, test error and generalisation*.  When the validation error
begins to rise while the training error still falls, the model has started to
overfit and the run should be stopped.  This early stopping improves
performance substantially in many settings and costs nothing.

**Adaptive methods do not always generalise well.** 
As noted in Section *Implementations*, several studies have found that
Adam, RMSProp and AdaGrad reach a worse test error than well-tuned SGD with
momentum, particularly when the number of parameters exceeds the number of
data points.  It is not settled why adaptive methods train deep networks so
effectively yet generalise less well; the practical conclusion is that a
properly tuned SGD may match or beat them, and is worth trying.

**Check your gradients.** 
Before trusting an optimiser, verify the gradient it is given.  Compare the
analytical expression against a central finite difference,

$$
\frac{\partial C}{\partial\theta_j}
   \approx \frac{C(\bm{\theta}+h\bm{e}_j)-C(\bm{\theta}-h\bm{e}_j)}{2h},\tag{4.43}
$$

with $h$ around $10^{-5}$, or against automatic differentiation.  An optimiser
given a wrong gradient does not usually crash; it quietly converges to the
wrong answer, which is far harder to detect.

**Do not compare learning rates across optimisers.** 
Table 4.1 made the point: the same $\eta$ means different
things to plain SGD and to Adam, because the latter has already normalised the
gradient.  Each method must be tuned on its own scale before any comparison is
meaningful.


## Summary and the programs

The chapter has one argument, developed in stages.

Minimising a cost function is easy when the cost is convex, because by
Section *Convexity* any stationary point is then a global minimum, and
the least-squares and Ridge problems of Chapter 3 are convex.
Newton's method (4.8) solves such problems ideally, in a
single step for a quadratic, and is invariant to how the parameters are
scaled.  It costs $\bigO(p^{3})$ per step and is therefore unavailable for
large models.

Gradient descent costs $\bigO(p)$ and pays for it in iterations.  The
decoupled error recursion (4.19) explained exactly why: the
step size is bounded above by $2/\lambda_{\max}$ for stability while progress
is governed by $\lambda_{\min}$, so the iteration count scales with the
condition number $\kappa$ of the Hessian -- and by
Eq. (4.22) with the *square* of the condition number
of the design matrix.  This single number connects the chapter to the two
before it: it is the quantity of Section *Vector and matrix norms* that governs
numerical accuracy, the quantity of Section *Statistical properties of the least-squares estimator* that
governs the variance of the fitted parameters, and now the quantity that
governs how long training takes.  Standardising features and adding a Ridge
penalty improve all three at once.

The remedies then form a sequence, each recovering a little more of what
Newton's method has and gradient descent lacks.  Momentum accumulates a
discounted history of gradients, amplifying consistent directions by
$1/(1-\gamma)$ and cancelling oscillatory ones, which improves the rate from
$\kappa$ to $\sqrt{\kappa}$.  Stochastic gradients replace the full sum by a
minibatch, trading a worse rate per iteration for a far better rate per unit
of computation, and adding noise that helps escape poor local minima.  The
adaptive methods build a diagonal estimate of curvature from the second moment
of the gradient: AdaGrad by an unbounded sum, which eventually stalls; RMSProp
by an exponential moving average, which does not; and Adam by combining
RMSProp with momentum and correcting the initialisation bias with the factors
$1-\beta_i^{t}$ derived in Eq. (4.40).

The demonstration of Table 4.1 was deliberately
unflattering, and its two lessons should be kept.  A learning rate is not
comparable across optimisers, because the adaptive methods have already
normalised the gradient before $\eta$ multiplies it.  And on a small, convex,
well-conditioned problem the adaptive machinery buys nothing and can cost a
great deal: these methods exist for large, ill-conditioned, non-convex
landscapes, which is where we shall meet them from Chapter 5 onwards.

Finally, none of this is usable without gradients, and
Section *Automatic differentiation* showed that they need not be derived by hand.
Reverse-mode automatic differentiation returns the gradient with respect to
all $p$ parameters for the price of about two function evaluations, and it is
the reason the models of the later chapters can be written down at all.

The complete programs are collected in `doc/BookML/BookPrograms`:

- `gradient_descent.py` -- plain and momentum gradient descent on
   $f(x)=x^{2}$ and on the least-squares and Ridge problems of
   Section *Gradient descent for linear regression*, with the analytical solutions for
   comparison, and the empirical verification of the stability
   bound (4.20) and the rate (4.21).
- `stochastic_gradient.py` -- minibatch SGD with a varying number
   of batches, the time-decay schedule (4.30), and a
   comparison of convergence per iteration against convergence per unit
   of computation.
- `adaptive_optimizers.py` -- AdaGrad, RMSProp and Adam as in
   Section *Implementations*, the learning-rate sweep of
   Table 4.1, and the same comparison repeated on a
   deliberately ill-conditioned design matrix where the adaptive methods
   win decisively.
- `autodiff_examples.py` -- `autograd` and `JAX`
   versions of the gradients used above, with the finite-difference
   gradient check of Eq. (4.43).

Each file runs as a script and reproduces the numbers quoted in this chapter.
An executable version is available as a Jupyter notebook in the accompanying
Jupyter-book.


## Exercises

### Warm-up exercises

1. **Convexity from the definition.**
   Show that $f(x)=x^{2}$ is convex on $\mathbb{R}$ using
   Eq. (4.1).  Hint: show that
   $\lambda f(x)+(1-\lambda)f(y)-f(\lambda x+(1-\lambda)y)\ge0$ for all
   $x,y$ and $\lambda\in[0,1]$.
2. **Convexity from the second-order condition.**
   Using the second-order condition of Section *Convexity*, show that
   (a) $f(x)=e^{x}$ is convex on $\mathbb{R}$;
   (b) $g(x)=-\ln(x)$ is convex on $(0,\infty)$.
3. **Compositions.**
   Let $f(x)=x^{2}$ and $g(x)=e^{x}$.  Show that $f(g(x))$ and $g(f(x))$ are
   convex on $\mathbb{R}$.  Show more generally that if $f$ is any convex
   function then $h(x)=e^{f(x)}$ is convex.
4. **Norms are convex.**
   A norm satisfies $f(\alpha\bm{x})=|\alpha|f(\bm{x})$ and
   $f(\bm{x}+\bm{y})\le f(\bm{x})+f(\bm{y})$.  Using only these two properties
   and Eq. (4.1), show that a norm is convex.  Deduce that
   both the Ridge and the Lasso cost functions of Chapter 3 are
   convex.
5. **The stability bound (numerical).**
   For the least-squares problem of Section *Gradient descent for linear regression*:
   (a) compute the eigenvalues of the Hessian (4.14) and hence
   $\lambda_{\max}$, $\lambda_{\min}$ and $\kappa$;
   (b) run gradient descent for $\eta$ just below and just above
   $2/\lambda_{\max}$ and confirm Eq. (4.20);
   (c) measure the number of iterations needed to reach a fixed accuracy for
   several $\eta$, and check that the minimum occurs near
   $\eta^{*}=2/(\lambda_{\max}+\lambda_{\min})$.
6. **Conditioning and convergence (numerical).**
   Construct design matrices with condition numbers
   $\kappa_2(\bm{X})=10,10^{2},10^{3}$, for instance by rescaling the columns.
   (a) Measure the iteration count of gradient descent to fixed accuracy for
   each, and check the linear scaling with $\kappa$ predicted by
   Eq. (4.21).
   (b) Repeat with momentum and verify the $\sqrt{\kappa}$ scaling of
   Eq. (4.26).
   (c) Repeat after standardising the columns, and comment.
7. **Momentum as a damped oscillator.**
   Consider the one-dimensional quadratic $C(\theta)=\tfrac{1}{2}\lambda\theta^{2}$.
   (a) Write the momentum update (4.23) as a two-term recurrence
   in $\theta$.
   (b) Find the values of $\eta$ and $\gamma$ for which the recurrence has
   complex roots, and interpret them as oscillation.
   (c) Verify Eq. (4.25) numerically by applying momentum to
   a constant gradient.
8. **Minibatch noise (numerical).**
   For a fixed $\bm{\theta}$, compute the minibatch gradient many times for
   batch sizes $M=1,4,16,64$ and measure the standard deviation of its
   components.  Verify the $1/\sqrt{M}$ scaling predicted by
   Section *The central limit theorem*, and discuss what it implies about the cost of
   reducing gradient noise.
9. **AdaGrad stalls (numerical).**
   Reproduce the $\eta=0.01$ column of Table 4.1.
   (a) Plot the effective learning rate $\eta/\sqrt{r_{t,j}}$ against $t$ for
   each coordinate and show that it decays towards zero.
   (b) Estimate the total distance the iterate can still travel after $t$
   steps, and compare with the distance remaining to the minimum.
   (c) Repeat with RMSProp and explain, using
   Eq. (4.37), why the decay does not occur.
10. **Adam's bias correction (numerical).**
   (a) Implement Adam with and without the bias
   correction (4.41) and compare the first twenty steps.
   (b) For a constant gradient $g$, verify Eq. (4.40)
   numerically.
   (c) Show that with $\beta_2=0.999$ the uncorrected $\sqrt{v_1}$ is smaller
   than the true root mean square gradient by a factor of about $32$, and
   explain what that does to the first step.

### Project-style exercise: optimisers on an ill-conditioned problem

The comparison in Table 4.1 was made on a problem too easy
to distinguish the methods.  The purpose of this exercise is to build one that
does, and thereby to see the arguments of this chapter operate.

**Part a: build the landscape.** 
Generate a polynomial regression problem from the Franke function of
Section *A complete example: the Franke function*, or from a one-dimensional polynomial of degree ten
with the Vandermonde design matrix (3.3).  Compute
$\kappa_2(\bm{X})$ and $\kappa_2(\bm{X}^{T}\bm{X})$ and confirm
Eq. (4.22).  Then produce a second version with the columns
standardised and compare the two condition numbers.

**Part b: plain gradient descent.** 
Implement gradient descent with the analytical
gradient (4.13).  Determine empirically the largest stable
$\eta$ and compare with Eq. (4.20).  Plot the cost against
iteration for several $\eta$ on both the raw and the standardised problem.

**Part c: momentum and stochasticity.** 
Add momentum and confirm the improvement in iteration count.  Then implement
minibatch SGD with the time-decay schedule (4.30), and compare
convergence per iteration and per gradient evaluation.  Which comparison is
the fair one, and why?

**Part d: the adaptive methods.** 
Implement AdaGrad, RMSProp and Adam.  For each, sweep $\eta$ over several
orders of magnitude and record the best result; present the outcome as a table
like Table 4.1.  Discuss which methods are sensitive to
$\eta$ and which are not, and relate your finding to whether the method
normalises the gradient before applying $\eta$.

**Part e: the diagonal approximation and its limits.** 
Construct two problems with the same condition number, one whose Hessian is
diagonal and one whose narrow valley runs at $45$ degrees to the coordinate
axes -- the second is obtained from the first by an orthogonal rotation of the
features, which by Section *Orthogonal transformations* leaves $\kappa$ unchanged.
Run all the methods on both.  Explain, using the final notebox of
Section *Adam*, why the adaptive methods help on the first and not on
the second, and what this says about the difference between a diagonal
approximation to $\bm{H}$ and the real thing.

**Part f: automatic differentiation.** 
Repeat part b using `autograd` or `JAX` instead of the
hand-derived gradient, and verify that the two agree to machine precision.
Then apply the gradient check (4.43) and compare the accuracy
of the finite difference with that of automatic differentiation as $h$ is
varied over many orders of magnitude.  Explain the shape of the resulting
curve using the discussion of round-off error in
Section *Vector and matrix norms*.
